# 🎧 Dimensionador WFM — v1
**Herramienta de dimensionamiento para call center inbound 24/7.**

Flujo: cargas tu previsión y tu archivo de agentes → calcula dimensionamiento Erlang del mes, chequeo de capacidad, ocupación (horas muertas) y right-sizing de plantilla → exporta a Excel.

**Cómo usar:** *Entorno de ejecución → Ejecutar todo* (Runtime → Run all). Cuando te lo pida, sube tus dos archivos. Todos los parámetros se editan en la celda de abajo.

## ⚙️ Parámetros (edita aquí)

In [ ]:
# Instala el optimizador (solo la primera vez por sesión)
!pip install ortools -q

AHT          = 420     # tiempo medio de manejo (segundos)
SLA_OBJ      = 0.80    # nivel de servicio objetivo (0.80 = 80%)
TARGET_SEG   = 20      # responder dentro de X segundos (80/20)
INTERVALO_MIN= 60      # duración del intervalo (60, 30 o 15)
AUSENTISMO   = 0.15    # 15%
TURNO_HORAS  = 9       # duración del turno
PROD_HORAS   = 8.0     # horas productivas por turno (9h menos descansos)
MES          = 6       # mes a procesar (6 = junio)

# Descansos por país (minutos dentro de un turno de 9h = 540 min)
BREAK_MIN   = {"Colombia": 40, "Espana": 75}   # Col: 30+10  |  Esp: 30 + 5/hora x9
CENTRO_PAIS = {"BOGOTA": "Colombia", "SEVILLA": "Espana", "BARCELONA": "Espana"}

## 🧮 Motor de cálculo (Erlang C / A)

In [ ]:
import pandas as pd, math, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from ortools.sat.python import cp_model

def erlang_b(a, c):
    b = 1.0
    for k in range(1, a + 1):
        b = (c * b) / (k + c * b)
    return b

def erlang_c(a, c):
    b = erlang_b(a, c); r = c / a
    return b / (1 - r + r * b)

def nivel_servicio(a, c, aht, obj):
    if a <= c: return 0.0
    return 1 - erlang_c(a, c) * math.exp(-(a - c) * obj / aht)

def agentes_necesarios(carga, aht, obj, sla):
    if carga <= 0: return 0
    a = int(carga) + 1
    while nivel_servicio(a, carga, aht, obj) < sla:
        a += 1
    return a

def nivel_atencion(agentes, carga, aht, paciencia_seg):
    if agentes <= 0: return 0.0
    lam = carga / aht; mu = 1.0 / aht; theta = 1.0 / paciencia_seg
    pj = 1.0; suma = 1.0
    for j in range(1, agentes + 1):
        pj = pj * lam / (j * mu); suma += pj
    lq = 0.0; k = 1
    while True:
        pj = pj * lam / (agentes * mu + k * theta); suma += pj; lq += k * pj
        if pj < 1e-15: break
        k += 1
    return 1 - theta * (lq / suma) / lam

print("Motor listo.")

## 1️⃣ Cargar previsión y desdoblar

In [ ]:
from google.colab import files
print("Sube tu archivo de PREVISIÓN (ej. PREV_JUNIO.xlsx):")
subido_prev = files.upload()

In [ ]:
nombre = list(subido_prev.keys())[0]
raw = pd.read_excel(nombre)

fechas = raw.iloc[0]                                   # fila 0 = fecha de cada columna
datos = raw.drop(index=0)
datos = datos[datos["Unnamed: 0"] != "total"]
datos = datos.rename(columns={"Unnamed: 0": "intervalo"})

largo = datos.melt(id_vars="intervalo", var_name="columna", value_name="volumen")
largo["fecha"] = largo["columna"].map(fechas)
largo = largo[largo["fecha"].apply(lambda d: isinstance(d, pd.Timestamp))]
largo["volumen"] = pd.to_numeric(largo["volumen"], errors="coerce")
largo = largo.dropna(subset=["volumen"])
largo = largo[largo["fecha"].dt.month == MES]
largo["intervalo"] = largo["intervalo"].astype(int)
largo = largo[["fecha", "intervalo", "volumen"]].sort_values(["fecha", "intervalo"])

print("Filas:", len(largo), "| Días:", largo["fecha"].dt.day.nunique())
largo.head()

## 2️⃣ Dimensionamiento del mes

In [ ]:
dur = INTERVALO_MIN * 60
largo["carga"]   = largo["volumen"] * AHT / dur
largo["agentes"] = largo.apply(lambda f: agentes_necesarios(f["carga"], AHT, TARGET_SEG, SLA_OBJ), axis=1)

resumen_dia = largo.groupby(largo["fecha"].dt.day)["agentes"].sum()
print("Agente-horas por día:"); print(resumen_dia)
print("\nTotal agente-horas del mes:", int(largo["agentes"].sum()))

## 3️⃣ Chequeo de capacidad (cargar agentes)

In [ ]:
print("Sube tu archivo de AGENTES (ej. AGENTES.xlsx):")
subido_ag = files.upload()

In [ ]:
nombre_ag = list(subido_ag.keys())[0]
ag = pd.read_excel(nombre_ag)
m = ag[(ag["MODO"] == "MULTISKILL") & (ag["ESTADO"] == "ACTIVO")].copy()

m["pais"] = m["CENTRO"].map(CENTRO_PAIS)
m["prod"] = m["pais"].map(lambda p: (TURNO_HORAS * 60 - BREAK_MIN[p]) / (TURNO_HORAS * 60))
m["horas_prod_mes"] = m["Jornada"] * (30 / 7) * m["prod"]

capacidad = m["horas_prod_mes"].sum() * (1 - AUSENTISMO)
demanda   = largo["agentes"].sum()

print("Agentes Multiskill activos:", len(m), "|", m["CENTRO"].value_counts().to_dict())
print("Demanda (h en teléfono):", int(demanda))
print("Capacidad disponible (h, con ausentismo):", int(capacidad))
print("Cobertura: %.0f%%" % (capacidad / demanda * 100))

## 4️⃣ Ocupación por hora (horas muertas)

In [ ]:
ocup = largo.groupby("intervalo").apply(
    lambda g: g["carga"].sum() / g["agentes"].sum() * 100 if g["agentes"].sum() > 0 else 0)

print("Ocupación global: %.0f%%" % (largo["carga"].sum() / largo["agentes"].sum() * 100))

colores = ["#e63946" if v < 35 else "#2a9d8f" for v in ocup]
plt.figure(figsize=(11, 4))
plt.bar(ocup.index.astype(str), ocup.values, color=colores)
plt.axhline(85, color="gray", ls="--", lw=1, label="Ocupación sana (~85%)")
plt.title("Ocupación media por hora — rojo = hora muerta (<35%)")
plt.xlabel("Hora"); plt.ylabel("Ocupación (%)"); plt.ylim(0, 100); plt.legend()
plt.tight_layout(); plt.show()

## 5️⃣ Right-sizing (5 días/semana + días libres)

In [ ]:
porfecha = largo.groupby("fecha")["agentes"].sum()
dow = porfecha.groupby(porfecha.index.dayofweek).mean()
req_dia = {d: math.ceil(dow[d] / PROD_HORAS) for d in range(7)}

nombres = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
for d in range(7):
    print(nombres[d], "->", req_dia[d], "agentes")

model = cp_model.CpModel()
patrones = {off: [d for d in range(7) if d not in {off, (off + 1) % 7}] for off in range(7)}
x = {p: model.NewIntVar(0, 1000, f"p{p}") for p in patrones}
for d in range(7):
    model.Add(sum(x[p] for p, trabaja in patrones.items() if d in trabaja) >= req_dia[d])
model.Minimize(sum(x[p] for p in patrones))
solver = cp_model.CpSolver(); solver.Solve(model)
head = sum(solver.Value(x[p]) for p in patrones)

print("\nHeadcount mínimo:", head, "| con ausentismo:", math.ceil(head / (1 - AUSENTISMO)))
print("Tienes:", len(m), "Multiskill")

## 6️⃣ Exportar resultados a Excel

In [ ]:
with pd.ExcelWriter("dimensionamiento_mes.xlsx") as w:
    largo.to_excel(w, sheet_name="Detalle", index=False)
    resumen_dia.rename("agente_horas").to_excel(w, sheet_name="ResumenDia")
    ocup.round(1).rename("ocupacion_%").to_excel(w, sheet_name="Ocupacion")

files.download("dimensionamiento_mes.xlsx")
print("¡Exportado!")